In [ ]:
import json
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'

In [ ]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database-6',
    value_deserializer=lambda v: json.loads(v.decode('utf-8'))
)


In [15]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [5]:
cur.execute(
        """CREATE TABLE processed_events (
            lpep_pickup_datetime TIMESTAMP,
            lpep_dropoff_datetime TIMESTAMP,
            PULocationID INTEGER,
            DOLocationID INTEGER,
            passenger_count INTEGER,
            trip_distance NUMERIC(10, 2),
            tip_amount NUMERIC(10, 2),
            total_amount NUMERIC(10, 2)
        )""")

DuplicateTable: relation "processed_events" already exists


In [19]:
ride = next(consumer)
ride.value

{'lpep_pickup_datetime': '2025-10-01 00:21:47',
 'lpep_dropoff_datetime': '2025-10-01 00:24:37',
 'PULocationID': 247,
 'DOLocationID': 69,
 'passenger_count': 1.0,
 'trip_distance': 0.7,
 'tip_amount': 1.7,
 'total_amount': 10.0}

In [11]:
import math
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    passenger_count = None if (ride['passenger_count'] is None or math.isnan(ride['passenger_count'])) else int(ride['passenger_count'])
    cur.execute(
        """INSERT INTO processed_events
           (lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, DOLocationID, passenger_count, trip_distance, tip_amount, total_amount)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (ride['lpep_pickup_datetime'], ride['lpep_dropoff_datetime'], ride['PULocationID'], ride['DOLocationID'], passenger_count, ride['trip_distance'], ride['tip_amount'], ride['total_amount'])
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

KeyboardInterrupt: 

In [ ]:
count = 0
for message in consumer:
    if message.value['trip_distance'] > 5:
        count += 1

print(f'Number of rides with distance  > 5: {count}')

KeyboardInterrupt: 